# Differentially Private LoRA Fine-Tuning of Causal Language Models

This tutorial fine-tunes GPT-2 with LoRA on E2E NLG using Opacus, PEFT, and Transformers. It compares a non-private baseline with three DP-LoRA configurations and reports BLEU, perplexity, GPU memory usage, and training throughput.

## Prerequisites

- Familiarity with DP-SGD and Opacus's `PrivacyEngine`. See the [text classifier tutorial](https://github.com/meta-pytorch/opacus/blob/main/tutorials/building_text_classifier.ipynb).
- Familiarity with [LoRA](https://arxiv.org/abs/2106.09685).

## Environment and runtime

The recorded runs use one Kaggle T4 GPU. Each configuration trains for two epochs and takes about 38 minutes. Training all four takes about 2.5 hours, excluding setup and evaluation. The installation cell below specifies the package version ranges used for this notebook.


## Why combine DP-SGD with LoRA?

DP-SGD clips per-example gradients and adds Gaussian noise to their sum. LoRA freezes the base model and trains low-rank adapters. In this notebook, the adapters contain 589,824 trainable parameters, about 0.47% of the model.

Training fewer parameters reduces the storage needed for per-example gradients. For a fixed clipping threshold and noise multiplier, it does not reduce the noise standard deviation per coordinate.

## The task: E2E NLG

[E2E NLG](https://arxiv.org/abs/1706.09254) is a structured-data-to-text task. Inputs are meaning representations such as `name[The Vaults], eatType[pub], priceRange[more than £30]`; outputs are restaurant descriptions. Several human-written descriptions may correspond to the same meaning representation.


## 1. Install dependencies

The following cell installs Opacus, PEFT, Transformers, and the dataset and evaluation libraries. These are version ranges rather than exact pins; the next cell prints the installed versions.


In [1]:
!pip install --quiet \
    'opacus>=1.6.0' \
    'peft>=0.18,<0.19' \
    'transformers>=5.0' \
    'accelerate' \
    'datasets' \
    'evaluate' \
    'sacrebleu'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00


## 2. Check the environment

Check the installed package versions and CUDA availability. Enable a GPU accelerator before running the training configurations.


In [2]:
import torch
import opacus
import peft
import transformers
import datasets

print(f'torch        = {torch.__version__}  (CUDA: {torch.cuda.is_available()})')
print(f'opacus       = {opacus.__version__}')
print(f'peft         = {peft.__version__}')
print(f'transformers = {transformers.__version__}')
print(f'datasets     = {datasets.__version__}')

if torch.cuda.is_available():
    print(f'device       = {torch.cuda.get_device_name(0)}')


torch        = 2.10.0+cu128  (CUDA: True)
opacus       = 1.6.0
peft         = 0.18.1
transformers = 5.0.0
datasets     = 5.0.0
device       = Tesla T4


## 3. Imports and device

Import the training libraries, set the PyTorch random seed, and select the device. A fixed seed does not ensure identical results across hardware and library versions.


In [3]:
import math
import time
from dataclasses import dataclass, field
from typing import Optional

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from opacus import PrivacyEngine
from datasets import load_dataset
import evaluate

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')


Using device: cuda


## 4. Load E2E NLG

Load the upstream training and development CSV files with pandas and convert them to a `DatasetDict`. This avoids relying on Hub dataset loading scripts. The columns are renamed to `meaning_representation` and `target`.

Each training row pairs one meaning representation with one reference description. For BLEU evaluation, the development references are grouped by meaning representation.


In [4]:
import pandas as pd
from datasets import Dataset, DatasetDict

E2E_TRAIN_URL = 'https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/trainset.csv'
E2E_DEV_URL   = 'https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/devset.csv'

df_train = pd.read_csv(E2E_TRAIN_URL).rename(columns={'mr': 'meaning_representation', 'ref': 'target'})
df_val   = pd.read_csv(E2E_DEV_URL).rename(columns={'mr': 'meaning_representation', 'ref': 'target'})

ds = DatasetDict({
    'train': Dataset.from_pandas(df_train),
    'validation': Dataset.from_pandas(df_val),
})
print(ds)
print()
print('--- Sample (train) ---')
print('MR:    ', ds['train'][0]['meaning_representation'])
print('Target:', ds['train'][0]['target'])


DatasetDict({
    train: Dataset({
        features: ['meaning_representation', 'target'],
        num_rows: 42061
    })
    validation: Dataset({
        features: ['meaning_representation', 'target'],
        num_rows: 4672
    })
})

--- Sample (train) ---
MR:     name[The Vaults], eatType[pub], priceRange[more than £30], customer rating[5 out of 5], near[Café Adriatic]
Target: The Vaults pub near Café Adriatic has a 5 star rating.  Prices start at £30.


### Select validation examples

Use the full training split, about 42,000 rows. For evaluation, select 100 development meaning representations with all their references for BLEU, and 200 development rows for perplexity.


In [5]:
# Use the full E2E NLG train split. For evaluation, group the dev split by
# meaning representation: E2E provides several human references per MR, and the
# official metric protocol scores generations against all of them.
EVAL_MRS = 100

ds_train_small = ds['train']
ds_val_small = ds['validation'].shuffle(seed=42).select(range(200))  # for perplexity

refs_by_mr = df_val.groupby('meaning_representation')['target'].apply(list)
eval_mrs = refs_by_mr.sample(n=EVAL_MRS, random_state=42)
print(f'Train examples: {len(ds_train_small)}')
print(f'Eval MRs: {len(eval_mrs)} (refs per MR: min={eval_mrs.str.len().min()}, '
      f'max={eval_mrs.str.len().max()})')


Train examples: 42061
Eval MRs: 100 (refs per MR: min=4, max=37)


## 5. Tokenize for causal-LM training

Format each row as `'{MR} -> {target}'`, then tokenize and pad or truncate it to 128 tokens. GPT-2's EOS token is also used for padding.

The labels are copies of the input IDs. The causal-LM loss therefore includes the prompt, target, and padding positions. The perplexity reported below uses this same loss and should not be interpreted as target-only perplexity.


In [6]:
MODEL_NAME = 'gpt2'
MAX_SEQ_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

PROMPT_TEMPLATE = '{mr} ->'

def tokenize_example(example):
    """Format as 'MR -> target', tokenize, pad to fixed length.

    Labels copy input_ids, so the loss includes prompt, target, and padding
    positions rather than only the target span.
    """
    prompt = PROMPT_TEMPLATE.format(mr=example['meaning_representation'])
    target = ' ' + example['target'] + tokenizer.eos_token
    full_text = prompt + target

    enc = tokenizer(
        full_text,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        truncation=True,
    )
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': enc['input_ids'],
    }

ds_train_tok = ds_train_small.map(tokenize_example, remove_columns=ds_train_small.column_names)
ds_val_tok = ds_val_small.map(tokenize_example, remove_columns=ds_val_small.column_names)
ds_train_tok.set_format('torch')
ds_val_tok.set_format('torch')
print(f'Tokenized train: {ds_train_tok}')
print(f'Tokenized val:   {ds_val_tok}')


07/20/2026 11:48:48:WARNING:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/42061 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenized train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 42061
})
Tokenized val:   Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


## 6. Model setup and sampling

### A. Move the model before adding adapters

Call `model.to(device)` before `get_peft_model()`. This is the workaround confirmed in [#820](https://github.com/meta-pytorch/opacus/issues/820) for the reported LoRA weight-update failure.

```python
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model = get_peft_model(model, config)
# Configure the optimizer and PrivacyEngine after adding the adapters.
```

### B. Enable training mode before attaching the privacy engine

Opacus requires the model to be in training mode when `make_private_with_epsilon()` is called. Call `model.train()` explicitly before that step.

### C. Sampling and privacy accounting

The default Poisson sampler can produce empty batches. In the GPT-2 setup used here, an empty batch causes a reshape error in the forward pass. The notebook sets `poisson_sampling=False` to retain the original shuffled DataLoader, which uses nonempty batches for this dataset.

The accountant assumes Poisson sampling. Its reported ε therefore does not establish the same privacy guarantee for this shuffled DataLoader. The table reports the accountant's output for these experiments, not a verified bound for the sampling scheme used. See [Chua et al., ICML 2024](https://arxiv.org/abs/2403.17673) for the distinction between privacy accounting under Poisson sampling and shuffling.


## 7. Run configuration

Define the model, optimizer, and privacy settings for each run.


In [7]:
@dataclass
class RunConfig:
    name: str
    lora: bool
    dp: bool
    lr: float = 1e-4
    batch_size: int = 8
    epochs: int = 2
    target_epsilon: float = 8.0
    target_delta: float = 1e-5
    max_grad_norm: float = 1.0
    lora_r: int = 16
    lora_alpha: int = 32


## 8. Build the model

Load GPT-2, move it to the selected device, and add LoRA adapters to the attention `c_attn` modules. All four configurations below enable LoRA.


In [8]:
def build_model_for_config(cfg: RunConfig):
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

    # DP-full path: swap GPT-2's transformers.Conv1D modules with nn.Linear so opacus's
    # per-sample-gradient hooks attach correctly. opacus has a registered fix for Conv1D.
    # No-op for the LoRA path (LoRA wraps Conv1D with its own A/B Linear modules, which
    # opacus already handles).
    if cfg.dp and not cfg.lora:
        from opacus.validators import ModuleValidator
        model = ModuleValidator.fix(model)
        print(f'[{cfg.name}] Applied ModuleValidator.fix() (Conv1D -> nn.Linear)')

    model = model.to(device)  # <-- BEFORE PEFT, per #820

    if cfg.lora:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=0.0,
            target_modules=['c_attn'],
        )
        model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'[{cfg.name}] trainable={trainable:,} ({100*trainable/total:.2f}% of {total:,})')
    return model


## 9. Training

Create an AdamW optimizer and a shuffled DataLoader. For DP configurations, attach the privacy engine with `poisson_sampling=False`, as described in Section 6.

Record training time, processed tokens, peak allocated GPU memory, and the accountant's final ε. Throughput counts all input tokens, including padding. The timer covers training, excluding model loading and evaluation.


In [9]:
def train_one_run(cfg: RunConfig, train_ds):
    print(f'\n=== Training: {cfg.name} ===')
    model = build_model_for_config(cfg)
    model.train()  # opacus's validator requires train mode before make_private_with_epsilon

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr,
    )
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)

    privacy_engine = None
    if cfg.dp:
        privacy_engine = PrivacyEngine(accountant='rdp')
        # poisson_sampling=False: uniform-without-replacement batches; see Pattern C
        # above for why, including the caveat on the accountant's analysis.
        model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=train_loader,
            target_epsilon=cfg.target_epsilon,
            target_delta=cfg.target_delta,
            epochs=cfg.epochs,
            max_grad_norm=cfg.max_grad_norm,
            poisson_sampling=False,
        )
        print(f'[{cfg.name}] noise_multiplier={optimizer.noise_multiplier:.4f}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_start = time.perf_counter()
    tokens_processed = 0

    model.train()
    step = 0
    loss_history = []
    for epoch in range(cfg.epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_history.append(loss.item())
            tokens_processed += batch['input_ids'].numel()
            if step % 500 == 0:
                print(f'  epoch {epoch} step {step:5d}  loss={loss.item():.4f}')
            step += 1

    t_total = time.perf_counter() - t_start
    peak_mem_gb = (torch.cuda.max_memory_allocated() / 1024**3) if torch.cuda.is_available() else 0.0
    final_epsilon = privacy_engine.get_epsilon(delta=cfg.target_delta) if cfg.dp else float('inf')

    return {
        'config': cfg,
        'model': model,
        'mean_loss': sum(loss_history) / len(loss_history),
        'final_loss': loss_history[-1],
        'tokens_per_sec': tokens_processed / t_total,
        'peak_mem_gb': peak_mem_gb,
        'wall_clock_sec': t_total,
        'epsilon': final_epsilon,
        'steps_completed': step,
    }


## 10. Evaluate BLEU and perplexity

Generate one description for each of the 100 selected development meaning representations using beam search with `num_beams=5`. Compute corpus-level SacreBLEU against all references associated with each selected meaning representation.

Compute perplexity separately from the mean loss over the 200 tokenized development rows. As described in Section 5, this loss includes prompt and padding tokens. The evaluation subset and loss definition must be taken into account when comparing with other results.


In [10]:
bleu_metric = evaluate.load('sacrebleu')

def evaluate_run(model, eval_mrs_series, val_ds_tok, max_new_tokens=64):
    """Multi-reference corpus BLEU with beam decoding, plus perplexity.

    eval_mrs_series: pandas Series indexed by meaning_representation, values are
    lists of human references (from the grouped dev split).
    """
    gen_model = model._module if hasattr(model, '_module') else model
    gen_model.eval()

    preds, refs = [], []
    max_refs = eval_mrs_series.str.len().max()

    with torch.no_grad():
        for mr, references in eval_mrs_series.items():
            prompt = PROMPT_TEMPLATE.format(mr=mr)
            prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
            gen = gen_model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                num_beams=5,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            pred = tokenizer.decode(gen[0, prompt_ids.shape[1]:], skip_special_tokens=True).strip()
            preds.append(pred)
            # sacrebleu requires the same reference count per prediction; pad by
            # repeating the first reference (standard practice, does not change BLEU)
            padded = references + [references[0]] * (max_refs - len(references))
            refs.append(padded)

        # perplexity on the tokenized validation split
        val_losses = []
        for i in range(min(200, len(val_ds_tok))):
            ex = val_ds_tok[i]
            out = gen_model(
                input_ids=ex['input_ids'].unsqueeze(0).to(device),
                attention_mask=ex['attention_mask'].unsqueeze(0).to(device),
                labels=ex['labels'].unsqueeze(0).to(device),
            )
            val_losses.append(out.loss.item())

    bleu = bleu_metric.compute(predictions=preds, references=refs)['score']
    perplexity = math.exp(sum(val_losses) / len(val_losses))
    return {'bleu': bleu, 'perplexity': perplexity, 'n_eval': len(preds)}


## 11. Configurations to compare

Compare a non-private LoRA baseline with three DP-LoRA configurations: a higher learning rate, a larger batch size, and a higher clipping threshold. The latter two use the baseline learning rate.

Clipping limits each example's gradient norm. The learning rate scales the optimizer update, while batch size and clipping threshold also affect the noisy gradient estimate. A larger clipping threshold permits larger gradients but also increases the noise standard deviation at a fixed noise multiplier. For discussion of private language-model fine-tuning and parameter-efficient training, see [Li et al., ICLR 2022](https://arxiv.org/abs/2110.05679) and [Yu et al., ICLR 2022](https://arxiv.org/abs/2110.06500).

The noise multiplier is calibrated separately for each DP configuration. Changing batch size also changes the number of updates per epoch, so this comparison does not isolate clipping as the only cause of utility differences.


In [11]:
# batch_size=64 for the larger-batch ablation exceeds 16 GB on a T4; 32 is the
# largest power-of-two that fits alongside opacus's per-sample gradients here.
CONFIGS = [
    RunConfig(name='non-DP LoRA',      lora=True, dp=False, lr=1e-4, batch_size=8,  epochs=2),
    RunConfig(name='DP LoRA lr=5e-4',  lora=True, dp=True,  lr=5e-4, batch_size=8,  epochs=2),
    RunConfig(name='DP LoRA bs=32',    lora=True, dp=True,  lr=1e-4, batch_size=32, epochs=2),
    RunConfig(name='DP LoRA clip=3.0', lora=True, dp=True,  lr=1e-4, batch_size=8,  epochs=2, max_grad_norm=3.0),
]


## 12. Run the configurations

Run two epochs per configuration. The recorded training times are about 38 minutes each on a T4. Each model is evaluated before the next configuration begins.


In [12]:
results = []
for cfg in CONFIGS:
    # Each configuration is isolated: if one runs out of memory on your GPU
    # (the larger-batch setting is the usual culprit), the sweep keeps going
    # and the comparison table reports whatever completed.
    try:
        train_result = train_one_run(cfg, ds_train_tok)
        eval_result = evaluate_run(train_result['model'], eval_mrs, ds_val_tok)
        results.append({**train_result, **eval_result})
        print(f'[{cfg.name}] BLEU={eval_result["bleu"]:.2f}  PPL={eval_result["perplexity"]:.2f}')
        del train_result['model']
    except torch.cuda.OutOfMemoryError:
        print(f'[{cfg.name}] SKIPPED: out of memory. Reduce batch_size for this configuration.')
    finally:
        torch.cuda.empty_cache() if torch.cuda.is_available() else None



=== Training: non-DP LoRA ===


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy b

[DP LoRA clip=3.0] trainable=589,824 (0.47% of 125,029,632)
[DP LoRA clip=3.0] noise_multiplier=0.3828


sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


  epoch 0 step     0  loss=8.7529
  epoch 0 step   500  loss=6.8720
  epoch 0 step  1000  loss=2.8173
  epoch 0 step  1500  loss=2.4631
  epoch 0 step  2000  loss=2.0345
  epoch 0 step  2500  loss=1.9668
  epoch 0 step  3000  loss=1.7959
  epoch 0 step  3500  loss=1.7770
  epoch 0 step  4000  loss=1.9407
  epoch 0 step  4500  loss=1.6403
  epoch 0 step  5000  loss=1.8761
  epoch 1 step  5500  loss=1.5938
  epoch 1 step  6000  loss=1.5857
  epoch 1 step  6500  loss=1.5342
  epoch 1 step  7000  loss=1.2777
  epoch 1 step  7500  loss=1.3622
  epoch 1 step  8000  loss=1.2682
  epoch 1 step  8500  loss=1.2796
  epoch 1 step  9000  loss=1.0052
  epoch 1 step  9500  loss=1.1905
  epoch 1 step 10000  loss=1.2610
  epoch 1 step 10500  loss=0.9629
[DP LoRA clip=3.0] BLEU=38.90  PPL=3.08


## 13. Comparison table

In [13]:
def fmt_eps(e):
    return 'no DP' if e == float('inf') else f'{e:.2f}'

print(f'{"Config":<14} | {"BLEU":>6} | {"PPL":>7} | {"tok/s":>8} | {"peak GB":>7} | {"sec":>6} | {"ε":>6}')
print('-' * 75)
for r in results:
    print(f'{r["config"].name:<14} | {r["bleu"]:>6.2f} | {r["perplexity"]:>7.2f} | '
          f'{r["tokens_per_sec"]:>8.0f} | {r["peak_mem_gb"]:>7.2f} | '
          f'{r["wall_clock_sec"]:>6.1f} | {fmt_eps(r["epsilon"]):>6}')


Config         |   BLEU |     PPL |    tok/s | peak GB |    sec |      ε
---------------------------------------------------------------------------
non-DP LoRA    |  68.38 |    1.50 |     4776 |    2.17 | 2254.3 |  no DP
DP LoRA lr=5e-4 |  47.13 |    2.02 |     4704 |    2.65 | 2288.8 |   8.00
DP LoRA bs=32  |  44.88 |    2.81 |     4652 |    8.16 | 2314.5 |   7.99
DP LoRA clip=3.0 |  38.90 |    3.08 |     4705 |    3.66 | 2288.3 |   8.00


## 14. Results

Recorded run on a Kaggle T4: GPT-2 with LoRA on `c_attn`, two epochs over the full training split, and multi-reference BLEU with beam-5 decoding over 100 development meaning representations.

| Config | BLEU | PPL | tok/s | Peak GB | Time | ε |
|---|---|---|---|---|---|---|
| non-DP LoRA (lr 1e-4) | 68.38 | 1.50 | 4776 | 2.17 | 2254 s | no DP |
| DP LoRA (lr 5e-4) | 47.13 | 2.02 | 4704 | 2.65 | 2289 s | 8.00 |
| DP LoRA (lr 1e-4, batch 32) | 44.88 | 2.81 | 4652 | 8.16 | 2315 s | 7.99 |
| DP LoRA (lr 1e-4, clip 3.0) | 38.90 | 3.08 | 4705 | 3.66 | 2288 s | 8.00 |

The ε column contains the accountant's output at δ = 1e-5, subject to the sampling limitation in Section 6. These values are not verified privacy bounds for the shuffled DataLoader.

### Learning rate, batch size, and clipping threshold

Among these runs, the DP configuration with learning rate 5e-4 has the highest BLEU, 47.13. Increasing batch size to 32 at learning rate 1e-4 gives 44.88. Increasing the clipping threshold to 3.0 gives 38.90. The non-private baseline scores 68.38.

These are single runs for each configuration. They do not establish a general ranking of the hyperparameters or measure variation across seeds.

### Increasing batch size affects memory usage

The batch-32 run reports peak allocated memory of 8.16 GiB, compared with 2.65 GiB for the batch-8 run at learning rate 5e-4. The table's GB column is computed in GiB (`bytes / 1024**3`).

A caveat on reported memory utilization: configurations run sequentially in one process. The two batch-8 DP runs report different peaks, 2.65 and 3.66 GiB. This experiment does not isolate the cause of that difference, so it should not be attributed to the clipping threshold alone.

### Comparison with other experiments

These results use a 100-meaning-representation subset of the development split. Comparisons with published results require matching the data split, model, training schedule, decoding settings, metric implementation, and privacy assumptions.


## 15. References

- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [Deep Learning with Differential Privacy](https://arxiv.org/abs/1607.00133)
- [E2E NLG Challenge dataset](https://arxiv.org/abs/1706.09254) and the [dataset repository](https://github.com/tuetschek/e2e-dataset)
- [Large Language Models Can Be Strong Differentially Private Learners](https://arxiv.org/abs/2110.05679)
- [Differentially Private Fine-tuning of Language Models](https://arxiv.org/abs/2110.06500)
- [How Private are DP-SGD Implementations?](https://arxiv.org/abs/2403.17673)
- [Opacus #820: LoRA weight-update issue](https://github.com/meta-pytorch/opacus/issues/820)
